# NB109 - Demo & verification of the newest RMT-PPAD model (8+ images + video)

Qualitative verification of the **newest** RMT-PPAD lane+detection model: the
`laneiou_both_full` run resumed in NB108, whose **best.pt is epoch 37** (curve-F1
0.621, curveIoU 0.752, detection mAP50 0.835; the NB108 length fix recovered the
mean lane length 0.094 -> 0.103). The aux drivable/lane-seg heads are
training-only and dropped at eval, so what you see is the true deployable model
(detection boxes + CLR polyline lanes).

**What it produces:**
1. **>= 8 validation images** - ground truth vs. best.pt prediction, saved as
   individual annotated PNGs (`rmt_ppad_pred_<stem>.png`) plus a combined grid,
   for dropping straight into the report's RMT-PPAD demonstration figure.
2. **An annotated video** - the model run frame-by-frame over a clip, written as
   an MP4 with a measured FPS overlay (same video approach as Stage-1 NB07).

**Where outputs go:** the run's checkpoint folder on Drive,
`.../training_runs/checkpoints/laneiou_both_full/demo/` (set by `DEMO_DIR`).

> If your newest `best.pt` lives under a different run folder, change `RUN_NAME`
> in Cell 1. The input video is auto-detected from
> `/content/drive/MyDrive/EcoCAR/video/input.mp4` (or any `.mp4` under an EcoCAR
> `video/` folder); change `VIDEO_IN` in Cell 5 if yours is elsewhere.

**Flow:** Cell 1 mount+deps+paths -> Cell 2 stage weights + val images -> Cell 3
load helpers (model/infer/draw) -> Cell 4 render >=8 image demos -> Cell 5
annotate a video.


### Cell 1: Mount Drive + deps + locate the NEWEST checkpoint

In [ ]:
import os, sys, subprocess
from pathlib import Path

os.environ['PYTHONIOENCODING'] = 'utf-8'
if not Path('/content/drive').exists():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing {REPO_ROOT} -- verify Drive sync.')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

for _pkg in ('addict', 'yapf'):
    try: __import__(_pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pkg])
for _m in ('mmcv',):
    try: __import__(_m)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _m])
try:
    import cv2  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'])

MIG       = Path(REPO_ROOT) / 'stage2/rmt_ppad_migration'
RMT       = MIG / 'vendor/RMT-PPAD'
DATASETS  = Path('/content/drive/MyDrive/EcoCAR/datasets')
FULL      = Path('/content/bdd_dataset')

# === The model under test ===========================================
# NB108 resumed the `laneiou_both_full` run; its best.pt is epoch 37 (the newest
# model). If your newest best.pt is under a different folder, change RUN_NAME.
CKPT_ROOT = Path('/content/drive/MyDrive/EcoCAR/training_runs/checkpoints')
RUN_NAME  = 'laneiou_both_full'
CKPT_DIR  = CKPT_ROOT / RUN_NAME
# All demo outputs (images + annotated video) go HERE, next to the weights:
DEMO_DIR  = CKPT_DIR / 'demo'
DEMO_DIR.mkdir(parents=True, exist_ok=True)

print('[ok] repo =', MIG)
print('[ok] checkpoint dir:', CKPT_DIR, '| exists:', CKPT_DIR.exists())
for w in ('best.pt', 'last.pt'):
    print(f'   {w}:', (CKPT_DIR / w).exists())
print('[ok] demo outputs ->', DEMO_DIR)
if not CKPT_DIR.exists():
    print('[hint] folders present under', CKPT_ROOT, ':',
          sorted(p.name for p in CKPT_ROOT.glob("*"))[:30] if CKPT_ROOT.exists() else '<root missing>')


### Cell 2: Get weights + a handful of val images onto /content/

The .pt files are read straight from Drive. Val images: if the full subset
isn't already on /content/ (a fresh session), extract just the val images +
lane_targets + labels from NB102's complete-dataset tar (fast, one file). We
only need a few dozen images to look at, but the tar is the simplest source.


In [ ]:
import tarfile

# Weights (read from Drive directly).
BEST = CKPT_DIR / 'best.pt'
LAST = CKPT_DIR / 'last.pt'
have_best, have_last = BEST.exists(), LAST.exists()
if not (have_best or have_last):
    raise FileNotFoundError(f'No best.pt/last.pt under {CKPT_DIR} - check the run synced to Drive.')
print('[weights] best:', have_best, ' last:', have_last)

# Val images + GT (for the GT panel). Reuse /content/bdd_dataset if present.
val_img = FULL / 'images/val2017'
if val_img.exists() and any(val_img.glob('*.jpg')):
    print(f'[data] using existing {val_img}')
else:
    tar = DATASETS / 'bdd_complete_labels_70k.tar.gz'
    curve = DATASETS / 'bdd100k_clrkd_curve.tar'
    # complete-labels tar gives labels+lane_targets+drivable; the curve tar has images.
    if curve.exists():
        print(f'[data] extracting val images from {curve.name} (this has the jpgs)...', flush=True)
        with tarfile.open(curve) as tf:
            members = [m for m in tf.getmembers()
                       if ('/val' in m.name or '/val2017' in m.name) and m.name.endswith('.jpg')]
            # take first 60 val jpgs only - we just need samples
            members = members[:60]
            for m in members:
                tf.extract(m, '/content/_curve_scratch')
        # normalize into FULL/images/val2017
        scratch = Path('/content/_curve_scratch')
        dst = val_img; dst.mkdir(parents=True, exist_ok=True)
        for jp in scratch.rglob('*.jpg'):
            (dst / jp.name).write_bytes(jp.read_bytes())
        print(f'[data] staged {sum(1 for _ in dst.glob("*.jpg"))} val images -> {dst}')
    else:
        raise FileNotFoundError(
            f'No val images on /content/ and {curve} not found. Run NB101/NB102 '
            'first (they extract /content/bdd_dataset), or place the curve tar on Drive.')

# GT labels/lane_targets (optional - only for the GT panel). Pull from complete tar.
lt = FULL / 'lane_targets/val2017'
if not (lt.exists() and any(lt.glob('*.pt'))):
    tar = DATASETS / 'bdd_complete_labels_70k.tar.gz'
    if tar.exists():
        print(f'[data] extracting GT labels from {tar.name}...', flush=True)
        with tarfile.open(tar) as tf:
            tf.extractall(FULL)
        print('[data] GT lane_targets/labels extracted')
    else:
        print('[warn] no complete-labels tar -> GT panel will show image only')
print('[ok] data staged')


### Cell 3: Load helpers - model, raw inference, per-lane decode, draw

Builds the model with the SAME path the validator uses (vendor-first import,
fuse disabled). `infer(model, img)` returns decoded lanes + detection boxes in
the 640 frame. `draw(...)` overlays them. Lanes are decoded with the curve-F1
metric's own `_decode_row`, so panels == what the metric scored.


In [ ]:
import importlib.util, numpy as np, cv2, torch

def _ensure_vendor_first(rmt_root):
    p = str(Path(rmt_root).resolve())
    sys.path[:] = [x for x in sys.path if 'ultralytics' not in x.lower()]
    if p in sys.path: sys.path.remove(p)
    sys.path.insert(0, p)
    for k in list(sys.modules):
        if k == 'ultralytics' or k.startswith('ultralytics.'):
            del sys.modules[k]

# curve-F1 decoder (the proven per-lane decode used by the metric)
_spec = importlib.util.spec_from_file_location('lcf1', MIG/'P7_validator/tools/lane_curve_f1.py')
lcf1 = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(lcf1)

S = 640
os.environ['USE_AUX_SEG'] = '1'; os.environ.setdefault('AUX_SEG_CLASSES', '2')  # aux row weights need the submodule to load
os.environ.setdefault('LANE_MATCH', 'hungarian'); os.environ.setdefault('LANE_DIFF_CLAMP', '100')
_ensure_vendor_first(RMT)
from ultralytics import MTDETR

# CRITICAL: the checkpoint was pickled with references to the migration shim
# module NAMES (p5_lane_losses, p4_lane_seg_head, aux_segmentors). When loading
# from a .pt (not a YAML), torch.load's unpickler must find those names in
# sys.modules or it raises `ModuleNotFoundError: No module named 'p5_lane_losses'`
# (then Ultralytics tries `pip install p5_lane_losses`, which fails). Importing
# the three vendor shims runs their top-level `_import_*()` which calls
# `sys.modules.setdefault('<name>', mod)` -> registers every pickle name the
# checkpoint needs. Must happen AFTER _ensure_vendor_first (so the vendor
# ultralytics is the one on the path) and BEFORE MTDETR(weights).
import importlib as _il
for _shim in ('ultralytics.models.utils.lane_losses',     # -> p5_lane_losses
              'ultralytics.nn.modules.lane_head',          # -> p4_lane_seg_head
              'ultralytics.nn.modules.aux_seg_head'):      # -> aux_segmentors
    try:
        _il.import_module(_shim)
    except Exception as _e:  # noqa: BLE001
        print(f'[warn] could not import shim {_shim}: {_e}')
import sys as _sys
_missing = [n for n in ('p5_lane_losses', 'p4_lane_seg_head', 'aux_segmentors')
            if n not in _sys.modules]
print('[shims] pickle names registered:',
      'ALL OK' if not _missing else f'MISSING {_missing}')

def load_model(weights):
    m = MTDETR(str(weights))
    inner = m.model
    inner.fuse = lambda *a, **k: inner            # fuse() is an inference-speed no-op that trips MTDETR; skip it
    inner.is_fused = lambda *a, **k: True
    inner.eval()
    return inner

def _preprocess(img_bgr):
    im = cv2.resize(img_bgr, (S, S))
    t = torch.from_numpy(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)).permute(2,0,1).float()/255.0
    return t.unsqueeze(0)

@torch.no_grad()
def infer(model, img_bgr, conf=0.3, max_lanes=8, lane_topk=8):
    dev = next(model.parameters()).device
    x = _preprocess(img_bgr).to(dev)
    preds = model(x)
    # ---- detection boxes: preds[0] = (1, N, 4+nc) xywh (in 0..1 * imgsz handled below) ----
    boxes = []
    try:
        p0 = preds[0] if isinstance(preds, (list, tuple)) else preds
        bb, sc = p0[..., :4], p0[..., 4:]
        bb = bb[0]; sc = sc[0]
        conf_v, cls_v = sc.max(-1)
        keep = conf_v > conf
        bb = bb[keep] * S; conf_v = conf_v[keep]; cls_v = cls_v[keep]
        # xywh -> xyxy
        for (cx,cy,w,h), cf, cl in zip(bb.tolist(), conf_v.tolist(), cls_v.tolist()):
            boxes.append((int(cx-w/2), int(cy-h/2), int(cx+w/2), int(cy+h/2), cf, int(cl)))
    except Exception as e:
        print('[warn] box decode failed:', e)
    # ---- lanes: preds[1][5] = dict with eval-mode final_preds (1, num_priors, 78) ----
    lanes = []
    try:
        seg = preds[1][5]
        lo = seg.get('lane_output') if isinstance(seg, dict) else seg
        if isinstance(lo, dict): fp = lo['predictions_lists'][-1]
        elif isinstance(lo, (list, tuple)): fp = lo[-1]
        else: fp = lo
        rows = fp[0].detach().cpu().numpy()          # (num_priors, 78)
        order = np.argsort(-lcf1._softmax_pos(rows))[:lane_topk]
        for r in rows[order]:
            poly = lcf1._decode_row(r, S, 72, is_pred=True)   # (M,2) int in 640 frame, or None
            if poly is not None:
                lanes.append(poly)
    except Exception as e:
        print('[warn] lane decode failed:', e)
    return lanes, boxes

def draw(img_bgr, lanes, boxes, lane_color=(0,255,255), box_color=(255,255,0)):
    ov = cv2.cvtColor(cv2.resize(img_bgr, (S,S)), cv2.COLOR_BGR2RGB).copy()
    for (x1,y1,x2,y2,*rest) in boxes:
        cv2.rectangle(ov, (x1,y1),(x2,y2), box_color, 2)
    for poly in lanes:
        cv2.polylines(ov, [poly.reshape(-1,1,2)], False, lane_color, 3)
        cv2.circle(ov, tuple(poly[0]), 5, (255,0,0), -1)
    return ov

# GT decode (green) for the reference panel
NS=71; YS=np.arange(S,-1,-S/NS)[:72]
def gt_lanes(stem):
    tp = FULL/'lane_targets/val2017'/f'{stem}.pt'
    out=[]
    if tp.exists():
        tt=torch.load(tp, weights_only=True); t=(tt.numpy() if hasattr(tt,'numpy') else np.asarray(tt)).astype(np.float32)
        for row in t:
            if float(row[1])<0.5: continue
            st=max(0,min(int(round(float(row[2])*NS)),72)); L=max(0,min(int(round(float(row[5]))),72-st))
            if L<2: continue
            xs=np.asarray(row[6+st:6+st+L]); ys=YS[st:st+L]
            pts=[(int(round(x)),int(round(y))) for x,y in zip(xs,ys) if np.isfinite(x) and 0<=x<S]
            if len(pts)>=2: out.append(np.array(pts,np.int32))
    return out
def gt_boxes(stem):
    lp=FULL/'labels/val2017'/f'{stem}.txt'; out=[]
    if lp.exists():
        for ln in lp.read_text().splitlines():
            f=ln.split()
            if len(f)<5: continue
            cx,cy,w,h=(float(v) for v in f[1:5])
            out.append((int((cx-w/2)*S),int((cy-h/2)*S),int((cx+w/2)*S),int((cy+h/2)*S)))
    return out

print('[ok] helpers ready')


### Cell 4: Render >= 8 validation images (GT vs. best.pt) and save them

Two panels per image: ground truth (green lanes, red boxes) and the newest
best.pt prediction (yellow lanes, cyan boxes). Each prediction panel is also
saved as its own PNG in `DEMO_DIR` so you can drop the best ones straight into
the report's RMT-PPAD demonstration figure, alongside a combined grid.


In [ ]:
import matplotlib.pyplot as plt, random

N_SHOW = 8                                   # >= 8 demo images, per request
assert have_best, 'No best.pt found - cannot run the demo on the newest model.'
m_best = load_model(BEST)

val_dir = FULL/'images/val2017'
stems = sorted(p.stem for p in val_dir.glob('*.jpg'))
random.seed(7); pick = random.sample(stems, min(N_SHOW, len(stems)))
print('demo images:', pick)

fig, axes = plt.subplots(len(pick), 2, figsize=(12, 5*len(pick)))
if len(pick) == 1: axes = [axes]
saved = []
for r, st in enumerate(pick):
    img = cv2.imread(str(val_dir/f'{st}.jpg'))
    # GT panel
    g = draw(img, gt_lanes(st), [(*b,1,0) for b in gt_boxes(st)], lane_color=(0,255,0), box_color=(255,0,0))
    axes[r][0].imshow(g); axes[r][0].set_title(f'{st}  GT  (lanes={len(gt_lanes(st))}, boxes={len(gt_boxes(st))})', fontsize=9); axes[r][0].axis('off')
    # prediction panel (newest best.pt)
    L, B = infer(m_best, img)
    pred = draw(img, L, B)
    axes[r][1].imshow(pred); axes[r][1].set_title(f'best.pt (ep37)  lanes={len(L)} boxes={len(B)}', fontsize=9); axes[r][1].axis('off')
    # save the prediction panel on its own (BGR for cv2) for the report
    outp = DEMO_DIR / f'rmt_ppad_pred_{st}.png'
    cv2.imwrite(str(outp), cv2.cvtColor(pred, cv2.COLOR_RGB2BGR))
    saved.append(str(outp))
plt.tight_layout()
grid = DEMO_DIR / 'rmt_ppad_demo_grid.png'
plt.savefig(grid, dpi=90); plt.show()
print(f'[saved] grid -> {grid}')
print(f'[saved] {len(saved)} individual prediction PNGs -> {DEMO_DIR}')
for s in saved: print('   ', s)
print('READ: yellow pred-lanes should hug road markings; cyan boxes wrap vehicles. '
      'Pick the clearest few for the report figure.')


### Cell 5: Annotate a video with the newest model (saved to the checkpoint folder)

Runs best.pt frame-by-frame over `VIDEO_IN`, overlays predicted lanes (yellow) +
vehicle boxes (cyan), and writes an annotated MP4 to `DEMO_DIR` with a measured
FPS overlay. Mirrors the Stage-1 NB07 video approach (OpenCV read loop +
VideoWriter), adapted to the RMT-PPAD `infer`/`draw` helpers. The model runs at
640x640 (its train/eval resolution); each annotated frame is resized back to the
source resolution for the output video.


In [ ]:
import time, cv2, glob

# Input video on Drive. The clip lives at EcoCAR/video/input.mp4; try the known
# locations and, failing that, any .mp4 under an EcoCAR video/ folder.
_video_candidates = [
    '/content/drive/MyDrive/EcoCAR/video/input.mp4',
    '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/video/input.mp4',
]
VIDEO_IN = next((p for p in _video_candidates if Path(p).exists()), None)
if VIDEO_IN is None:
    hits = sorted(glob.glob('/content/drive/MyDrive/EcoCAR/video/*.mp4')) + \
           sorted(glob.glob('/content/drive/MyDrive/EcoCAR/**/video/*.mp4', recursive=True))
    VIDEO_IN = hits[0] if hits else None
if not VIDEO_IN:
    raise FileNotFoundError(
        'No input video found. Tried:\n  ' + '\n  '.join(_video_candidates) +
        '\nPlace your clip at /content/drive/MyDrive/EcoCAR/video/input.mp4 '
        '(or set VIDEO_IN to its path) and re-run.')
print(f'[video] using input: {VIDEO_IN}')

VIDEO_OUT = str(DEMO_DIR / 'rmt_ppad_demo.mp4')
MAX_FRAMES = 1200          # cap so a long clip doesn't run forever (~40s @30fps)
DRAW_FPS_TEXT = True

m = globals().get('m_best') or load_model(BEST)

cap = cv2.VideoCapture(VIDEO_IN)
src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 1280
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 720
n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
print(f'[video] in={VIDEO_IN}  {W}x{H} @ {src_fps:.1f}fps  frames~{n_total}')

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(VIDEO_OUT, fourcc, src_fps, (W, H))

n, infer_ms = 0, []
t_start = time.time()
while n < MAX_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break
    t0 = time.time()
    L, B = infer(m, frame)                       # decode lanes + boxes (640 frame)
    infer_ms.append((time.time() - t0) * 1000)
    ann_rgb = draw(frame, L, B)                  # RGB, 640x640
    ann_bgr = cv2.cvtColor(ann_rgb, cv2.COLOR_RGB2BGR)
    ann_bgr = cv2.resize(ann_bgr, (W, H))        # back to source resolution
    if DRAW_FPS_TEXT and infer_ms:
        fps_now = 1000.0 / max(1e-6, sum(infer_ms[-30:]) / len(infer_ms[-30:]))
        cv2.putText(ann_bgr, f'RMT-PPAD ep37  {fps_now:5.1f} FPS (model)',
                    (12, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2, cv2.LINE_AA)
    writer.write(ann_bgr)
    n += 1
    if n % 100 == 0:
        print(f'  ...{n} frames  (model {sum(infer_ms)/len(infer_ms):.1f} ms/frame)', flush=True)
cap.release(); writer.release()

wall = time.time() - t_start
mean_ms = sum(infer_ms) / max(1, len(infer_ms))
print(f'[done] wrote {n} annotated frames -> {VIDEO_OUT}')
print(f'[perf] model {mean_ms:.1f} ms/frame ({1000.0/max(1e-6,mean_ms):.1f} FPS, GPU, lanes+boxes decode incl.); '
      f'wall {wall:.1f}s end-to-end incl. video I/O')
print('NOTE: model FPS is the network forward + decode; end-to-end is lower due '
      'to CPU video read/encode (Stage-1 NB07 removed that bottleneck with a '
      'producer-consumer pipeline; here we keep it simple for a demo clip).')
